# 第15章 物理实验、模型拟合与模拟数据

这个 notebook 用一个极小 RC 放电教学数据，演示指数模型拟合、残差、Monte Carlo 参数扰动和经验多项式比较。

## 学习目标

- 读入实验型数据并检查变量范围
- 用指数衰减模型搜索最佳时间常数
- 计算残差和加权 chi-square
- 用 Monte Carlo 粗略估计参数不确定度
- 比较物理模型与经验模型的差别

In [ ]:
from __future__ import annotations

import csv
import math
import platform
import random
from pathlib import Path

DATA_PATH = Path('../../data/small/physics_model_fit_demo.csv')

with DATA_PATH.open(newline='', encoding='utf-8') as handle:
    rows = [
        {
            'sample_id': row['sample_id'],
            'time_s': float(row['time_s']),
            'voltage_v': float(row['voltage_v']),
            'sigma_v': float(row['sigma_v']),
            'split': row['split'],
            'experiment_type': row['experiment_type'],
        }
        for row in csv.DictReader(handle)
    ]

times = [row['time_s'] for row in rows]
voltages = [row['voltage_v'] for row in rows]
errors = [row['sigma_v'] for row in rows]

print(f'Loaded {len(rows)} experiment rows from {DATA_PATH.name}')
print('Time span [s]:', (min(times), max(times)))
print('Voltage range [V]:', (round(min(voltages), 3), round(max(voltages), 3)))
print('Typical sigma [V]:', round(sum(errors) / len(errors), 3))
print('Python version:', platform.python_version())


In [ ]:
def solve_linear_system(matrix, vector):
    augmented = [row[:] + [value] for row, value in zip(matrix, vector)]
    n = len(augmented)
    for col in range(n):
        pivot = max(range(col, n), key=lambda idx: abs(augmented[idx][col]))
        augmented[col], augmented[pivot] = augmented[pivot], augmented[col]
        pivot_value = augmented[col][col]
        if abs(pivot_value) < 1e-12:
            raise ValueError('Singular matrix in polynomial fit')
        for j in range(col, n + 1):
            augmented[col][j] /= pivot_value
        for row_idx in range(n):
            if row_idx == col:
                continue
            factor = augmented[row_idx][col]
            for j in range(col, n + 1):
                augmented[row_idx][j] -= factor * augmented[col][j]
    return [augmented[i][-1] for i in range(n)]

def fit_exponential_tau(samples, tau):
    weights = [1.0 / (row['sigma_v'] ** 2) for row in samples]
    basis = [math.exp(-row['time_s'] / tau) for row in samples]
    numerator = sum(weight * row['voltage_v'] * value for row, weight, value in zip(samples, weights, basis))
    denominator = sum(weight * value * value for weight, value in zip(weights, basis))
    v0 = numerator / denominator
    model = [v0 * value for value in basis]
    chi2 = sum(((row['voltage_v'] - pred) / row['sigma_v']) ** 2 for row, pred in zip(samples, model))
    residuals = [row['voltage_v'] - pred for row, pred in zip(samples, model)]
    return {'tau': tau, 'v0': v0, 'chi2': chi2, 'model': model, 'residuals': residuals}

tau_grid = [1.0 + 0.05 * i for i in range(41)]
fits = [fit_exponential_tau(rows, tau) for tau in tau_grid]
fits.sort(key=lambda item: item['chi2'])
best = fits[0]

print('Top exponential candidates:')
for candidate in fits[:5]:
    print('  tau={:.2f} s chi2={:.3f} V0={:.3f} V'.format(candidate['tau'], candidate['chi2'], candidate['v0']))


In [ ]:
def fit_quadratic(samples):
    normal = [[0.0] * 3 for _ in range(3)]
    rhs = [0.0, 0.0, 0.0]
    for row in samples:
        t = row['time_s']
        design = [1.0, t, t * t]
        weight = 1.0 / (row['sigma_v'] ** 2)
        for i in range(3):
            rhs[i] += weight * design[i] * row['voltage_v']
            for j in range(3):
                normal[i][j] += weight * design[i] * design[j]
    c0, c1, c2 = solve_linear_system(normal, rhs)
    model = [c0 + c1 * row['time_s'] + c2 * row['time_s'] * row['time_s'] for row in samples]
    chi2 = sum(((row['voltage_v'] - pred) / row['sigma_v']) ** 2 for row, pred in zip(samples, model))
    return {'coefficients': (c0, c1, c2), 'model': model, 'chi2': chi2}

poly_fit = fit_quadratic(rows)
print('Best exponential tau [s]:', round(best['tau'], 3))
print('Best exponential V0 [V]:', round(best['v0'], 3))
print('Best exponential chi2:', round(best['chi2'], 3))
print('Quadratic chi2:', round(poly_fit['chi2'], 3))
print('Quadratic coefficients:', tuple(round(value, 4) for value in poly_fit['coefficients']))


In [ ]:
degrees_of_freedom_exp = len(rows) - 2
degrees_of_freedom_poly = len(rows) - 3
reduced_chi2_exp = best['chi2'] / degrees_of_freedom_exp
reduced_chi2_poly = poly_fit['chi2'] / degrees_of_freedom_poly
print('Reduced chi-square (exponential) =', round(reduced_chi2_exp, 3))
print('Reduced chi-square (quadratic) =', round(reduced_chi2_poly, 3))


In [ ]:
positive_rows = [row for row in rows if row['voltage_v'] > 0.0]
log_times = [row['time_s'] for row in positive_rows]
log_voltages = [math.log(row['voltage_v']) for row in positive_rows]
x_mean = sum(log_times) / len(log_times)
y_mean = sum(log_voltages) / len(log_voltages)
slope = sum((x - x_mean) * (y - y_mean) for x, y in zip(log_times, log_voltages)) / sum((x - x_mean) ** 2 for x in log_times)
intercept = y_mean - slope * x_mean
tau_from_log_fit = -1.0 / slope
v0_from_log_fit = math.exp(intercept)
print('Log-linearized fit parameters:')
print('  tau ~', round(tau_from_log_fit, 3), 's')
print('  V0  ~', round(v0_from_log_fit, 3), 'V')
print('  note: this fit is convenient, but it changes the effective error model.')


In [ ]:
random.seed(42)
mc_taus = []
for _ in range(80):
    synthetic = []
    for row in rows:
        perturbed_voltage = random.gauss(row['voltage_v'], row['sigma_v'])
        synthetic.append({**row, 'voltage_v': perturbed_voltage})
    synthetic_fit = [fit_exponential_tau(synthetic, tau) for tau in tau_grid]
    synthetic_fit.sort(key=lambda item: item['chi2'])
    mc_taus.append(synthetic_fit[0]['tau'])

mc_mean_tau = sum(mc_taus) / len(mc_taus)
mc_std_tau = math.sqrt(sum((tau - mc_mean_tau) ** 2 for tau in mc_taus) / len(mc_taus))
print('Monte Carlo tau mean [s]:', round(mc_mean_tau, 3))
print('Monte Carlo tau std [s]:', round(mc_std_tau, 3))
print('Monte Carlo tau range [s]:', (round(min(mc_taus), 3), round(max(mc_taus), 3)))


In [ ]:
def forward_euler_decay(v0, tau, dt, n_steps):
    values = [v0]
    for _ in range(n_steps):
        values.append(values[-1] - (dt / tau) * values[-1])
    return values

dt = 0.4
simulated = forward_euler_decay(best['v0'], best['tau'], dt=dt, n_steps=len(rows) - 1)
print('First five forward-simulation voltages:')
for value in simulated[:5]:
    print('  ', round(value, 3))


In [ ]:
try:
    import matplotlib.pyplot as plt
except Exception as exc:
    print('matplotlib unavailable; skipped experiment plots:', exc)
else:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].errorbar(times, voltages, yerr=errors, fmt='o', color='tab:blue', capsize=2, label='data')
    axes[0].plot(times, best['model'], color='tab:orange', label='exp fit')
    axes[0].plot(times, poly_fit['model'], color='tab:green', linestyle='--', label='quadratic')
    axes[0].set_title('RC Discharge Fit')
    axes[0].set_xlabel('Time [s]')
    axes[0].set_ylabel('Voltage [V]')
    axes[0].legend()
    axes[1].axhline(0.0, color='black', linewidth=1.0)
    axes[1].scatter(times, best['residuals'], color='tab:red')
    axes[1].set_title('Exponential Residuals')
    axes[1].set_xlabel('Time [s]')
    axes[1].set_ylabel('Residual [V]')
    for ax in axes:
        ax.grid(alpha=0.25)
    fig.tight_layout()
    print('Displayed fit comparison and residuals.')


## 解释

这个最小示例把实验数据分析串成了一条完整链：先用物理模型定义参数，再用 chi-square 搜索最佳时间常数，随后检查残差，并通过 Monte Carlo 评估参数波动范围。二次多项式也许能在局部贴近数据，但只有指数模型中的时间常数可以直接回到 RC 放电机制本身。

In [ ]:
snapshot = {
    'dataset': DATA_PATH.name,
    'n_rows': len(rows),
    'best_tau_s': round(best['tau'], 3),
    'best_v0_v': round(best['v0'], 3),
    'best_chi2': round(best['chi2'], 3),
    'best_reduced_chi2': round(reduced_chi2_exp, 3),
    'quadratic_chi2': round(poly_fit['chi2'], 3),
    'mc_mean_tau_s': round(mc_mean_tau, 3),
    'mc_std_tau_s': round(mc_std_tau, 3),
    'python_version': platform.python_version(),
}

print('Physics-model snapshot:')
for key, value in snapshot.items():
    print(f'  {key}: {value}')


## V2 Evidence Record artifact

本章的记录重点是模型公式、参数意义、残差/chi-square 诊断、baseline 比较和物理解释边界。


In [ ]:
artifact_dir = Path('results')
artifact_dir.mkdir(parents=True, exist_ok=True)
evidence_path = artifact_dir / 'ch15_evidence_record.md'

residual_range = (round(min(best['residuals']), 4), round(max(best['residuals']), 4))
mc_tau_range = (round(min(mc_taus), 3), round(max(mc_taus), 3))
baseline_comparison = f"exponential chi2 {best['chi2']:.3f} vs quadratic chi2 {poly_fit['chi2']:.3f}"

record_text = f"""# Evidence Record - Ch15 Physics Model and Simulated Data

## Record Type

- Record type: model fit

## 1. Input

- Data / file path, preferably relative to project root: `{DATA_PATH}`
- Data source or generation method: AIforAstronomers teaching RC discharge table
- Script / notebook: `ch15_physics_experiment_simulation_data.ipynb`
- Code version / tag, if relevant: fill in the current Git commit or release tag when submitting

## 2. Operation

- Command or entry point: run notebook cells in order
- Key parameters: tau grid `{tau_grid[0]:.2f}` to `{tau_grid[-1]:.2f}` s in `{tau_grid[1] - tau_grid[0]:.2f}` s steps; Monte Carlo seed `42`; `{len(mc_taus)}` perturbations
- Random seed, if relevant: `42` for Monte Carlo perturbations
- Output directory: `results/`

## 3. Output

- Output file(s): `{evidence_path}`; optional display plot if matplotlib is available
- Short result summary: best tau `{best['tau']:.3f}` s; V0 `{best['v0']:.3f}` V; reduced chi-square `{reduced_chi2_exp:.3f}`; MC tau std `{mc_std_tau:.3f}` s
- Generated by which script/notebook: `ch15_physics_experiment_simulation_data.ipynb`

## 4. Check

- Check performed: compared physical exponential model with quadratic baseline, inspected residual range, computed reduced chi-square, ran Monte Carlo perturbations
- Evidence from the check: {baseline_comparison}; residual range `{residual_range}` V; MC tau range `{mc_tau_range}` s; log-linear tau `{tau_from_log_fit:.3f}` s

## 5. Limit

- Known limitation: grid search and Monte Carlo are coarse teaching approximations; log-linear fit changes the error model
- Selection / quality / uncertainty issue, if relevant: small teaching experiment with simplified Gaussian voltage errors and no instrument calibration model

## 6. Reuse in ML

- Sample / feature / label: rows can become time-voltage samples; fitted tau/V0/residual summaries could become features or labels in synthetic workflows
- Uncertainty / quality flag: per-row `sigma_v` is used in chi-square; MC spread gives a rough parameter uncertainty
- Preprocessing record: records model form, tau grid, baseline comparison, and simulation step
- Reproducibility evidence: records source table, seed, fitting settings, residual diagnostics, and limits

## Ch15 Fields

- model formula: `V(t) = V0 exp(-t / tau)`
- parameter meanings: `V0` initial voltage; `tau` decay time constant
- parameter units: `V0` in V; `tau` in s
- fitting method: weighted grid search over tau with analytic V0 for each tau
- residual plot: displayed if matplotlib is available; residual range `{residual_range}` V recorded here
- error metric / chi-square: best chi-square `{best['chi2']:.3f}`; reduced chi-square `{reduced_chi2_exp:.3f}`
- uncertainty estimate: Monte Carlo tau mean `{mc_mean_tau:.3f}` s, std `{mc_std_tau:.3f}` s
- baseline comparison: {baseline_comparison}; quadratic reduced chi-square `{reduced_chi2_poly:.3f}`
- physical interpretation limit: exponential tau maps to RC discharge; quadratic baseline may fit locally but has no direct mechanism here
"""

evidence_path.write_text(record_text, encoding='utf-8')
print('wrote Evidence Record:', evidence_path)
